# 01. Extraccion cruda de convocatorias SICOES

Notebook para concentrar solo la descarga cruda y el almacenamiento reproducible.

Salidas principales:
- `data/raw/sicoes_convocatorias_raw.jsonl`
- `data/raw/sicoes_convocatorias_raw.parquet`
- `data/raw/sicoes_convocatorias_raw.csv` como exportacion auxiliar local


In [1]:
import json
import math
import re
import time
from pathlib import Path

import pandas as pd
import requests
from tqdm.auto import tqdm

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATA_RAW_DIR = ROOT / "data" / "raw"
DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)

RAW_JSONL_PATH = DATA_RAW_DIR / "sicoes_convocatorias_raw.jsonl"
RAW_PARQUET_PATH = DATA_RAW_DIR / "sicoes_convocatorias_raw.parquet"
RAW_CSV_EXPORT_PATH = DATA_RAW_DIR / "sicoes_convocatorias_raw.csv"


In [2]:
BASE_URL = "https://www.sicoes.gob.bo"
PAGE_URL = f"{BASE_URL}/portal/contrataciones/busqueda/convocatorias.php?tipo=convNacional"
POST_URL = f"{BASE_URL}/portal/contrataciones/operacion.php"

HEADERS_GET = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/149 Safari/537.36",
}

HEADERS_POST = {
    "Accept": "application/json, text/javascript, */*; q=0.01",
    "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
    "Origin": BASE_URL,
    "Referer": PAGE_URL,
    "X-Requested-With": "XMLHttpRequest",
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/149 Safari/537.36",
}

session = requests.Session()


def get_token() -> str:
    response = session.get(PAGE_URL, headers=HEADERS_GET, timeout=30)
    response.raise_for_status()
    html_text = response.text

    patterns = [
        r'name=["\']token["\']\s+value=["\']([^"\']+)["\']',
        r'id=["\']token["\']\s+value=["\']([^"\']+)["\']',
        r'token["\']?\s*[:=]\s*["\']([a-f0-9]{64})["\']',
    ]

    for pattern in patterns:
        match = re.search(pattern, html_text, flags=re.IGNORECASE)
        if match:
            return match.group(1)

    raise ValueError("No se pudo encontrar el token en el HTML.")


def build_payload(draw: int, token: str) -> dict[str, str]:
    return {
        "token": token,
        "entidad": "",
        "objetoContrato": "",
        "publicacionDesde": "",
        "publicacionHasta": "",
        "presentacionPropuestasDesde": "",
        "presentacionPropuestasHasta": "",
        "cuce1": "",
        "cuce2": "",
        "cuce3": "",
        "cuce4": "",
        "cuce5": "",
        "cuce6": "",
        "r1": "11",
        "subasta": "",
        "bienes": "B",
        "obras": "O",
        "servicios": "S",
        "consultoria": "C",
        "tipo": "Simple",
        "operacion": "convNacional",
        "autocorrector": "",
        "nroRegistros": "10",
        "draw": str(draw),
        "start": "0",
        "length": "10",
        "captcha": "",
    }


def request_page(draw: int, token: str) -> dict:
    response = session.post(
        POST_URL,
        headers=HEADERS_POST,
        data=build_payload(draw, token),
        timeout=30,
    )
    response.raise_for_status()
    text = response.text.strip()
    if not text.startswith("{"):
        raise ValueError(f"La respuesta de draw={draw} no parece JSON.")
    return json.loads(text)


In [3]:
TOKEN = get_token()
test_page = request_page(1, TOKEN)

records_total = int(test_page.get("recordsTotal") or test_page.get("recordsFiltered") or 0)
page_size = len(test_page.get("data", [])) or 10
total_pages = math.ceil(records_total / page_size)

print({"records_total": records_total, "page_size": page_size, "total_pages": total_pages})


{'records_total': 1419, 'page_size': 10, 'total_pages': 142}


In [4]:
raw_records = []

for draw in tqdm(range(1, total_pages + 1), desc="Descargando paginas SICOES"):
    try:
        page_json = request_page(draw, TOKEN)
        for idx, raw in enumerate(page_json.get("data", [])):
            raw_records.append({"draw": draw, "row_in_page": idx, "raw_record": raw})
        time.sleep(0.15)
    except Exception as exc:
        print(f"Error en draw={draw}: {exc}")

len(raw_records)


Descargando paginas SICOES:   0%|          | 0/142 [00:00<?, ?it/s]

1419

In [5]:
with RAW_JSONL_PATH.open("w", encoding="utf-8") as f:
    for item in raw_records:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

raw_rows_expanded = []
for item in raw_records:
    raw = item["raw_record"]
    row = {
        "draw": item["draw"],
        "row_in_page": item["row_in_page"],
        "raw_record_json": json.dumps(raw, ensure_ascii=False),
    }
    if isinstance(raw, dict):
        for key, value in raw.items():
            row[f"raw_{key}"] = value
    else:
        for idx, value in enumerate(raw):
            row[f"raw_{idx}"] = value
    raw_rows_expanded.append(row)

raw_df = pd.DataFrame(raw_rows_expanded)
raw_df.to_parquet(RAW_PARQUET_PATH, index=False)
raw_df.to_csv(RAW_CSV_EXPORT_PATH, index=False, encoding="utf-8-sig")

print("Archivos generados:")
print("-", RAW_JSONL_PATH)
print("-", RAW_PARQUET_PATH)
print("-", RAW_CSV_EXPORT_PATH, "(auxiliar local)")
raw_df.head()


Archivos generados:
- /var/www/codigo/maestria_ia/umsa/diplomados_intermedios/dip_03/data/raw/sicoes_convocatorias_raw.jsonl
- /var/www/codigo/maestria_ia/umsa/diplomados_intermedios/dip_03/data/raw/sicoes_convocatorias_raw.parquet
- /var/www/codigo/maestria_ia/umsa/diplomados_intermedios/dip_03/data/raw/sicoes_convocatorias_raw.csv (auxiliar local)


,draw,row_in_page,raw_record_json,raw_ngcj3Y75,raw_HmCTpBsK,raw_GA9HJiCY,raw_2iKqCOEV,raw_Xg5wGtcV,raw_eQ9vSgMg,raw_hxefryPa,raw_PHKyMiDU,raw_v3IOV5OV,raw_vnRfzD4S,raw_TZHzvW6f,raw_t1ubouvP
0,1,0,"{""ngcj3Y75"": ""26-1803-00-1669918-1-1"", ""HmCTpB...",26-1803-00-1669918-1-1,Gobierno Autonomo Municipal De Riberalta,Bienes,CNC,Provision De Medicamentos E Insumos Medicos Pa...,Si,29/06/2026,03/07/2026,Vigente,,%3C%61%20%68%72%65%66%3D%27%23%27%20%6F%6E%63%...,<a onclick=irFicha('/portal/contrataciones/fic...
1,1,1,"{""ngcj3Y75"": ""26-1101-00-1668944-1-1"", ""HmCTpB...",26-1101-00-1668944-1-1,Gobierno Autonomo Municipal De Sucre,Bienes,CM,Adquisicion De Sardina En Conserva En Salsa De...,No,29/06/2026,02/07/2026,Vigente,"<a href='#' onClick=""descargarArchivo('KhznHa5...",%3C%61%20%68%72%65%66%3D%27%23%27%20%6F%6E%63%...,<a onclick=irFicha('/portal/contrataciones/fic...
2,1,2,"{""ngcj3Y75"": ""26-1119-00-1669680-1-1"", ""HmCTpB...",26-1119-00-1669680-1-1,Gobierno Autonomo Municipal De Camargo,Bienes,ANPP,Adquisición De Medicamentos E Insumos Médicos ...,Si,29/06/2026,09/07/2026,Vigente,"<a href='#' onClick=""descargarArchivo('0DQV71M...",%3C%61%20%68%72%65%66%3D%27%23%27%20%6F%6E%63%...,<a onclick=irFicha('/portal/contrataciones/fic...
3,1,3,"{""ngcj3Y75"": ""26-1101-00-1669911-1-1"", ""HmCTpB...",26-1101-00-1669911-1-1,Gobierno Autonomo Municipal De Sucre,Bienes,CM,Compra De Fideo Cortado Para El Programa Alime...,No,29/06/2026,02/07/2026,Vigente,"<a href='#' onClick=""descargarArchivo('5wEiTEK...",%3C%61%20%68%72%65%66%3D%27%23%27%20%6F%6E%63%...,<a onclick=irFicha('/portal/contrataciones/fic...
4,1,4,"{""ngcj3Y75"": ""26-0020-21-1664086-2-1"", ""HmCTpB...",26-0020-21-1664086-2-1,Armada Boliviana,Bienes,ANPP,"Adquisición De Materiales, Insumos, Repuestos ...",Si,29/06/2026,09/07/2026,Vigente,"<a href='#' onClick=""descargarArchivo('kmKhFgr...",%3C%61%20%68%72%65%66%3D%27%23%27%20%6F%6E%63%...,<a onclick=irFicha('/portal/contrataciones/fic...
